In [1]:
import pandas as pd
import os
from imblearn.under_sampling import RandomUnderSampler

# === 1. Load Split Datasets ===
split_dir = "../data/splits"
train_df = pd.read_csv(os.path.join(split_dir, "train.csv"))

print("Loaded split datasets:")
print(f" - train.csv: {len(train_df):,}")



Loaded split datasets:
 - train.csv: 4,453,834


In [2]:
# Basic Stats
n_pos = train_df["isFraud"].sum()
n_neg = (train_df["isFraud"] == 0).sum()
print(f"\nBefore downsampling:")
print(f"  Fraud cases:     {n_pos:,}")
print(f"  Non-fraud cases: {n_neg:,}")
print(f"  Fraud rate:      {train_df['isFraud'].mean():.5f}")

X_train = train_df.drop(columns=["isFraud"])
y_train = train_df["isFraud"]



Before downsampling:
  Fraud cases:     5,749
  Non-fraud cases: 4,448,085
  Fraud rate:      0.00129


In [3]:
# Define a function for downsampling
def downsample_dataset(X, y, ratio, random_state=42):
    n_pos = int(y.sum())
    n_neg = int((y == 0).sum())
    target_neg = min(ratio * n_pos, n_neg)

    rus = RandomUnderSampler(
        sampling_strategy={0: target_neg, 1: n_pos},
        random_state=random_state,
        replacement=False
    )
    X_ds, y_ds = rus.fit_resample(X, y)
    df_ds = pd.concat([X_ds, y_ds], axis=1).sample(frac=1, random_state=random_state).reset_index(drop=True)

    fraud_rate = df_ds["isFraud"].mean()
    print(f"\n=== AFTER DOWNSAMPLING (1:{ratio}) ===")
    print(f"Fraud: {df_ds['isFraud'].sum():,}, Non-fraud: {(df_ds['isFraud']==0).sum():,}")
    print(f"Fraud rate: {fraud_rate:.4f} (expected ≈ {1/(1+ratio):.4f})")
    print(f"Total samples: {len(df_ds):,}")

    return df_ds

# Create two downsampled datasets
train_ds_1to10 = downsample_dataset(X_train, y_train, ratio=10) #1:10
train_ds_1to5 = downsample_dataset(X_train, y_train, ratio=5) #1:5

# Save all versions
train_1to10_path = os.path.join(split_dir, "train_downsampled_1to10.csv")
train_1to5_path = os.path.join(split_dir, "train_downsampled_1to5.csv")

train_ds_1to10.to_csv(train_1to10_path, index=False)
train_ds_1to5.to_csv(train_1to5_path, index=False)

print("\n Saved downsampled training sets:")
print(f" - {train_1to10_path}: {len(train_ds_1to10):,} rows")
print(f" - {train_1to5_path}:  {len(train_ds_1to5):,} rows")



=== AFTER DOWNSAMPLING (1:10) ===
Fraud: 5,749, Non-fraud: 57,490
Fraud rate: 0.0909 (expected ≈ 0.0909)
Total samples: 63,239

=== AFTER DOWNSAMPLING (1:5) ===
Fraud: 5,749, Non-fraud: 28,745
Fraud rate: 0.1667 (expected ≈ 0.1667)
Total samples: 34,494

 Saved downsampled training sets:
 - ../data/splits/train_downsampled_1to10.csv: 63,239 rows
 - ../data/splits/train_downsampled_1to5.csv:  34,494 rows


In [4]:

# Representativeness check
def check_representativeness(original, downsampled, label="1:10"):
    print(f"\n=== REPRESENTATIVENESS CHECK ({label}) ===")
    # Fraud/non-fraud counts
    counts = pd.DataFrame({
        'Before': original['isFraud'].value_counts(),
        'After': downsampled['isFraud'].value_counts()
    }).rename(index={0: 'Non-Fraud', 1: 'Fraud'})
    print("\nFraud vs Non-Fraud Counts:")
    print(counts)

    # Transaction type distribution
    type_before = original['type'].value_counts(normalize=True) * 100
    type_after = downsampled['type'].value_counts(normalize=True) * 100
    type_compare = pd.concat([type_before, type_after], axis=1)
    type_compare.columns = ['Before (%)', 'After (%)']
    print("\nTransaction Type Distribution (Before vs After):")
    print(type_compare.round(2))

    # Numeric feature representativeness
    numeric_cols = original.select_dtypes(include=['float64', 'int64']).columns.drop('isFraud')
    desc_before = original[numeric_cols].describe().loc[['mean', 'std']]
    desc_after = downsampled[numeric_cols].describe().loc[['mean', 'std']]
    desc_diff = (desc_after - desc_before).round(3)

    print("\nNumeric Feature Means/STD Change (After - Before):")
    print(desc_diff.T)

check_representativeness(train_df, train_ds_1to10, label="1:10")
check_representativeness(train_df, train_ds_1to5, label="1:5")



=== REPRESENTATIVENESS CHECK (1:10) ===

Fraud vs Non-Fraud Counts:
            Before  After
isFraud                  
Non-Fraud  4448085  57490
Fraud         5749   5749

Transaction Type Distribution (Before vs After):
          Before (%)  After (%)
type                           
CASH_OUT       35.16      36.43
PAYMENT        33.82      30.98
CASH_IN        22.00      19.87
TRANSFER        8.38      12.13
DEBIT           0.65       0.59

Numeric Feature Means/STD Change (After - Before):
                      mean         std
step                11.224      11.892
amount          120474.649  432742.563
oldbalanceOrg    54531.449   36705.621
newbalanceOrig  -79688.766 -111177.523
oldbalanceDest  -19700.071  457715.732
newbalanceDest   37619.314  493125.182
isFlaggedFraud       0.000       0.013

=== REPRESENTATIVENESS CHECK (1:5) ===

Fraud vs Non-Fraud Counts:
            Before  After
isFraud                  
Non-Fraud  4448085  28745
Fraud         5749   5749

Transaction Type